# Chapter 14 &mdash; "You Call These Proofs?!" &mdash; On Informal Proof in Computability

**Concept 9 of the Chapter 14 decomposition:** *"You Call These Proofs?!" — On Informal Proof in Computability*

English plus pseudo-code is the accepted standard, and mechanical provers exist for when it is not.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14/Concept-Informal-Proof/Concept-Informal-Proof.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Computability proofs are written in **English with pseudo-code**, and that is the
professional standard, not a shortcut.

The reason is the Church&ndash;Turing thesis (Chapter 13, Concept 3): once you accept that
anything effectively calculable is TM-computable, a clear English description of a
procedure **is** a description of a TM. Writing the transition table adds pages and
subtracts clarity.

What such a proof must still do:

* say precisely **what is constructed**, and from what;
* show the construction is **computable** &mdash; a finite, mechanical recipe;
* show it has the **claimed property**, usually by cases.

When that is not enough &mdash; because the stakes are high or the argument is subtle &mdash;
mechanical theorem provers (Coq, Isabelle, Lean) will check it symbol by symbol.

## 2. Definitions

### An informal proof, and its obligations

In [ ]:
OBLIGATIONS = [
 ("construct", "state exactly what object is built and from what inputs"),
 ("computable", "argue the construction is a finite mechanical recipe"),
 ("correct",   "show the built object has the claimed property"),
]

PROOF = '''
CLAIM  If L1 and L2 are recursive then L1 union L2 is recursive.

PROOF  Let M1 and M2 be deciders for L1 and L2.  Build M:
         on input w:  run M1 on w;  if it accepts, accept.
                      run M2 on w;  if it accepts, accept.
                      reject.
       M is computable: it is a finite program calling two deciders.
       M halts on every input: M1 and M2 do, and M makes two calls.
       M accepts w iff w is in L1 or in L2.   QED
'''

### The same construction, executable

In [ ]:
def union_decider(d1, d2):
    def M(w):
        if d1(w): return True
        if d2(w): return True
        return False
    return M

## 3. Tests

The proof, as it would appear in a textbook.

In [ ]:
print(PROOF)
for what, why in OBLIGATIONS:
    print("  %-11s %s" % (what, why))

And the same construction, run.

In [ ]:
d1 = lambda w: w.startswith('1')
d2 = lambda w: w.endswith('0')
M  = union_decider(d1, d2)
from itertools import product
strs = [''.join(p) for k in range(1, 6) for p in product('01', repeat=k)]
assert all(M(w) == (d1(w) or d2(w)) for w in strs)
print("the constructed decider agrees with the specification on all %d strings"
      % len(strs))

**Why running it is not a proof**, and why the English is.

In [ ]:
print("the test covered %d strings; the claim is about infinitely many" % len(strs))
print()
print("The English argument covers them all because it reasons about the")
print("STRUCTURE of M: two terminating calls, then a decision.  That is what")
print("a proof does that a test cannot.")

Where informal proof goes wrong: the two failure modes.

In [ ]:
FAILURES = [("hand-waving computability",
             "'clearly we can compute f' -- when f needs to solve halting"),
            ("skipping a case",
             "'and similarly for the other case' -- when the other case differs")]
for a, b in FAILURES: print("  %-28s %s" % (a, b))
print()
print("Both are caught by writing out the obligations explicitly.")

And the escape hatch when the stakes are high.

In [ ]:
PROVERS = ["Coq", "Isabelle/HOL", "Lean", "ACL2", "PVS"]
print("mechanical checkers :", ', '.join(PROVERS))
print()
print("The four-colour theorem, seL4 and CompCert were checked this way.")
print("For a homework proof, English with the three obligations is the standard.")

## 4. Exercises


1. Rewrite the union proof for **intersection**. What changes?
2. Try it for RE languages instead of recursive ones. Where does the argument break?
3. Find a published proof that skips a case. (They exist.)

In [ ]:
# Your work for the exercises above.